# 158 — Resiliencia, idempotencia, rollback y recuperación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Las dependencias remotas fallan con normalidad estadística; los patrones de
estabilidad (Nygard, *Release It!*) convierten ese hecho en diseño:

- **Retry + backoff exponencial + jitter**:
  `espera_n = random(0, min(tope, base·2^n))` (full jitter, AWS). Solo para
  errores transitorios (timeout, 429, 503), con intentos acotados; sin jitter,
  los clientes reintentan sincronizados y re-tumban al servicio.
- **Circuit breaker**: máquina de estados por dependencia — *cerrado* (pasa y
  cuenta fallos), *abierto* (rechaza de inmediato, fail fast) y *semiabierto*
  (llamada de prueba). Protege al sistema entero de la falla en cascada; el
  retry solo protege a una petición.
- **Idempotencia**: `f(f(x)) = f(x)`. Prerrequisito de los reintentos: se
  implementa con una *idempotency key* por operación lógica que el servidor
  deduplica devolviendo el resultado original.
- **Rollback**: volver a un artefacto anterior versionado e inmutable; no
  deshace efectos ya producidos ni migraciones de esquema.
- **Saga** (Garcia-Molina y Salem, 1987): pasos T₁…Tₙ con compensaciones
  C₁…Cₙ; si falla Tₖ se ejecutan Cₖ₋₁…C₁ en orden inverso. Sin aislamiento
  ACID: consistencia eventual, y las compensaciones también deben ser
  idempotentes.


## 🧮 Ejemplo de referencia

Retry con `base = 0.5 s`, `tope = 30 s`, 4 intentos, full jitter:

```text
tras intento 0 → espera ~ U(0, 0.5 s)
tras intento 1 → espera ~ U(0, 1 s)
tras intento 2 → espera ~ U(0, 2 s)
intento 3 falla → error definitivo

peor caso acumulado = 3.5 s (+ 4 timeouts) · media ≈ 1.75 s
```

Breaker «5 fallos en 30 s, recuperación 60 s»: con la API caída, tras ~5
llamadas el breaker abre y el resto falla en <1 ms hacia el fallback, en vez
de pagar 3.5 s por petición. Saga «reservar viaje»: T₁ vuelo/C₁ cancelar,
T₂ hotel/C₂ cancelar, T₃ cobro/C₃ reembolso; si T₃ falla en definitivo se
ejecutan C₂ y C₁, con idempotency key por paso.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("workflow", seed=158)
show(result)


## Reflexión

1. Un equipo añadió reintentos «para robustez» a una llamada que crea pedidos y
   aparecieron pedidos duplicados: ¿qué faltó exactamente y en qué lado
   (cliente o servidor) se implementa?
2. Durante una caída parcial del proveedor de LLM, la latencia p99 de TODO tu
   sistema se multiplicó aunque la mayoría de peticiones no usaban el LLM:
   ¿qué antipatrón de Nygard describe esto y qué patrón lo corta?
3. ¿Por qué el rollback del prompt a la versión anterior puede no recuperar la
   calidad, y qué evidencia (clases 153-156) te permitiría distinguir la causa
   en horas y no en semanas?
